# A2-Lite 640×192 — full-dataset recovery and A2 distillation
Latest revision: **A2-LITE-FULL-RECOVERY-KD-2026-10-13-r1**

Same student: MobileNetV4 Medium, one visual encoder / one detection decoder, 256 channels, 50 queries, 640×192 input. Same teacher: frozen original A2 epoch130, full-resolution ONNX on CPU. No architecture changes, MonoDGP, Core ML, old runtime restore or CUDA extension build. Use a fresh GPU Colab session; A100 is preferred. No phone is needed.

The reviewed 512-frame pilot learned and KD improved AP, but pedestrian nearby recall fell from 58.55% to 51.85%. This experiment changes only data coverage and budget: all 3,712 Chen TRAIN frames, 10 fixed epochs per arm (37,120 updates), GT-only control versus GT plus the SAME selective A2 guidance. Both arms restart the identical original pruned weights, NOT the 16-frame diagnostic or either pilot endpoint.

All training images are used once per epoch, including naturally imbalanced classes and empty product-target frames. We do not claim balanced object counts. GT loss weights, class-balanced KD means, object matching, temperature 2, KD weight 0.25, learning rates, BatchNorm freezing, no augmentation/dropout/AMP, and deployed batch1/50-query training path remain as in the pilot. Optional dense depth-map loss and training query groups remain disabled.

Predeclared KD guardrail: all five AP metrics must not regress versus the matched GT control, mean moderate 3D AP must improve by at least 0.1 AP point, and nearby recall must not drop more than 2 absolute percentage points for EITHER class. Nearby recall uses the existing 2D-IoU matching protocol, not successful 3D reconstruction: score≥0.001, 2D IoU≥0.5, Vehicle≤40m / Pedestrian≤30m. A relative KD improvement does NOT mean the product targets passed. The comparison reports both separately.

There is no automatic geometry-only third arm, longer run, checkpoint selection or deployment. If recall still regresses, return the result before a separately reviewed geometry-only ablation. Validation does not choose training settings or checkpoints. Both fixed epoch10 endpoints are evaluated on all 3,769 VAL images.

Run sections 1–8 in order. This is 14.5× the pilot's optimization work and 7.25× its teacher-cache coverage; allow several hours, not minutes. Up to 512 hash-verified teacher records can be reused. New teacher records, training checkpoints and endpoint inference are resumable. After reset rerun sections 1–5, then resume 6. Checkpoints save every 128 steps; up to 127 interrupted steps may repeat. Return the ZIP even if a quality gate fails; do not lower gates or overwrite completed experiments.

## 1. Mount Drive, paths and all helpers
PILOT may be the completed recovery Drive folder or its original results ZIP. INITIALIZATION must be the original pruned checkpoint, which is not inside results ZIPs. No older notebook cells are required.

In [ ]:
# A2-LITE-FULL-RECOVERY-KD-2026-10-13-r1
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, re, shlex, subprocess, sys, zipfile
REVISION='A2-LITE-FULL-RECOVERY-KD-2026-10-13-r1'
RUN_ID='a2_lite_640_full_recovery_kd_r1'
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID):raise ValueError('Use lowercase letters, digits and underscores')
MOBILE_REPO=Path('/content/mobile_adas3d')
PILOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_recovery/a2_lite_640_recovery_kd_r1')
INITIALIZATION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite/a2_lite_e1_d1_cpu_screen_r1/a2_lite_initialization.pth')
COMPACT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_resolution/a2_lite_resolution_cpu_screen_r1')
TEACHER_EXPORT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2_m68_onnx_cpu/m68_a2_onnx_cpu_r3/export')
SPLITS=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
SOURCE_REPO=Path('/content/MonoDETR_A2LITE_FULL_'+RUN_ID)
DATASET=Path('/content/kitti_a2_lite_accuracy_full_'+RUN_ID)
CANDIDATES=[Path(p) for p in ('/content/kitti_a2_lite_accuracy_recovery_a2_lite_640_recovery_kd_r1','/content/kitti_a2_lite_accuracy_a2_lite_640_accuracy_r1','/content/kitti_m68','/content/kitti_m67','/content/kitti','/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti')]
OUTPUT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_full_recovery')/RUN_ID
PYTHON=Path(sys.executable);SCRIPT=MOBILE_REPO/'scripts/recover_a2_lite_full.py'
LOGS=OUTPUT/'colab_logs';LOGS.mkdir(parents=True,exist_ok=True)
READY=False;TINY_PASSED=False;KD_ALLOWED=False
def run_logged(command,name,cwd=None):
    command=list(map(str,command));path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',path,flush=True);tail=deque(maxlen=40)
    with path.open('w') as log:
        log.write(shlex.join(command)+'\n');log.flush()
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        finally:process.stdout.close()
        code=process.wait();log.write(f'\nEXIT CODE: {code}\n');log.flush()
    if code:raise RuntimeError(f'Exit {code}; full log={path}\n'+'\n'.join(tail))
    return path
def stage(name,*extra):
    if not READY and name!='bundle':raise RuntimeError('Run sections 1 and 2 first')
    return run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,name,'--output',OUTPUT,'--device','cuda',*extra],name,MOBILE_REPO)
print(REVISION,'\nOutput:',OUTPUT,'\nAll3712 train frames, two fixed10-epoch endpoints; no phone needed.')


## 2. Update main, dependencies and GPU check
Only ordinary Python dependencies are installed. Torch/CUDA are not replaced. On reset, restore package versions from THIS full-recovery manifest. The original pilot is read-only evidence, not an old virtual environment to restore.

In [ ]:
READY=False
if not MOBILE_REPO.exists():run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty:raise RuntimeError('Preserve tracked edits and use a clean main checkout.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only','origin','main'],'update',MOBILE_REPO)
if not SCRIPT.is_file():raise FileNotFoundError(SCRIPT)
for path in (PILOT,INITIALIZATION,COMPACT,TEACHER_EXPORT,SPLITS/'train.txt',SPLITS/'val.txt'):
    if not path.exists():raise FileNotFoundError(f'{path}: correct the path in section 1')
versions={'numpy':'2.1.3','timm':'1.0.27','onnxruntime':'1.30.0','pillow':'11.3.0','scipy':'1.16.3','opencv-python-headless':None}
manifest_path=OUTPUT/'full_recovery_manifest.json'
if manifest_path.exists():saved=json.loads(manifest_path.read_text())['cpu_environment']
elif PILOT.is_dir():saved=json.loads((PILOT/'recovery_manifest.json').read_text())['cpu_environment']
else:
    with zipfile.ZipFile(PILOT) as archive:
        matches=[n for n in archive.namelist() if n=='recovery_manifest.json' or n.endswith('/recovery_manifest.json')]
        if len(matches)!=1:raise RuntimeError('Expected one reviewed pilot manifest in ZIP')
        saved=json.loads(archive.read(matches[0]))['cpu_environment']
for name in versions:versions[name]=saved[name]
packages=[name+('=='+version if version else '') for name,version in versions.items()]+['pyyaml','scikit-image']
run_logged([PYTHON,'-m','pip','install','-q',*packages],'dependencies')
run_logged([PYTHON,'-c',"import torch,timm,onnxruntime as ort;assert torch.cuda.is_available(),'Select a fresh GPU Colab runtime';torch.cuda.init();assert timm.__version__=='1.0.27';assert ort.__version__=='1.30.0';print(torch.__version__,torch.cuda.get_device_name(0),'portable tensor training, no native extension')"],'runtime_probe')
READY=True


## 3. Tests, reviewed pilot and private data staging
Freeze exact reviewed pilot reports, original initialization, teacher, portable source and train/val data hashes. Use every TRAIN frame; do not class-filter or select from VAL. Old experiment files remain untouched.

In [ ]:
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite*recovery.py','-v'],'tests',MOBILE_REPO)
extra=[]
for candidate in CANDIDATES:extra += ['--candidate',candidate]
stage('prepare','--pilot',PILOT,'--initialization',INITIALIZATION,'--compact',COMPACT,'--teacher-export',TEACHER_EXPORT,'--repo',SOURCE_REPO,'--dataset',DATASET,'--split-dir',SPLITS,*extra)


## 4. Same initialization parity, GPU gradients and tiny learning check
On a disposable copy, reproduce all16 initial CPU fixtures and verify finite nonzero GPU gradients. Run the same 128-update/16-TRAIN-frame learning check. The tiny checkpoint is never used to initialize either larger arm. A failed check skips training but remains a reportable result.

In [ ]:
stage('smoke')
stage('tiny')
TINY=json.loads((OUTPUT/'tiny_fit.json').read_text());TINY_PASSED=bool(TINY['passed'])
print(json.dumps({k:TINY[k] for k in ('passed','gt_loss_ratio','geometry_error_ratio','geometry_ratio_per_class','evaluation_scope')},indent=2))
if not TINY_PASSED:print('STOP: return section8 ZIP. Do not lower the learning gate.')


## 5. Frozen A2 cache on all3712 TRAIN images
Reuse up to512 exact hash-verified old teacher records, rebinding their provenance to this new run without changing their values. Compute/verify remaining TRAIN guidance on CPU. No VAL guidance is cached. Actual encoded object counts and teacher coverage are reported; natural class imbalance is not hidden.

In [ ]:
KD_ALLOWED=False
if TINY_PASSED:
    stage('cache','--pilot',PILOT)
    CACHE=json.loads((OUTPUT/'teacher_cache_complete.json').read_text());KD_ALLOWED=bool(CACHE['kd_pilot_authorized'])
    print(json.dumps({k:CACHE[k] for k in ('images','encoded_gt_counts','qualified_teacher_matches','kd_pilot_authorized')},indent=2))
    if not KD_ALLOWED:print('No KD arm authorized: evaluate GT control only and return coverage evidence.')
else:print('Skipped: learning check failed.')


## 6. Matched full-data training — 10 fixed epochs each
Each role gets37,120 updates from the same original pruned weights. Same deterministic frame order, optimizer, rates and losses as the pilot; only all-train coverage and duration change. Checkpoint every128 updates. After a reset, rerun1–5 then this cell. Do not change RUN_ID, epochs or losses while resuming.

In [ ]:
if TINY_PASSED:
    stage('train','--role','control')
    if KD_ALLOWED:stage('train','--role','kd')
else:print('No training started; return the learning-check evidence.')


## 7. Full VAL endpoints and pedestrian-recall guardrail
Evaluate each fixed epoch10 checkpoint on all3769 VAL frames. No best-epoch selection. Product accuracy and nearby-recall gates are separate from relative KD benefit. A failed recall guardrail is a finding, not an execution error; no geometry-only third arm or deployment starts automatically.

In [ ]:
if TINY_PASSED:
    stage('evaluate','--role','control')
    if KD_ALLOWED:
        stage('evaluate','--role','kd')
        stage('compare')
        REPORT=json.loads((OUTPUT/'full_recovery_comparison.json').read_text())
        print(json.dumps({k:REPORT[k] for k in ('kd_minus_control','nearby_recall_delta','nearby_recall_guardrails','kd_benefit_guardrail_passed','all_product_accuracy_gates_passed','all_product_nearby_recall_gates_passed','next_review')},indent=2))
    else:print('GT control evaluated; no KD benefit is claimed.')
else:print('Validation skipped: learning check failed.')


## 8. Download results — also after a failure
Return a2_lite_full_recovery_results.zip. It contains metrics, predictions, cache identities and combined logs, not large student checkpoints/ONNX/images. Checkpoints stay on Drive for resume and a separately reviewed export. Passing this relative experiment does not qualify deployment.

In [ ]:
BUNDLE=OUTPUT/'a2_lite_full_recovery_results.zip'
try:
    if not SCRIPT.is_file():raise RuntimeError('Script unavailable; package readable logs directly')
    stage('bundle')
except Exception as error:
    print('Bundle helper unavailable:',error,'— packaging readable results/logs directly.')
    with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and 'assets' not in path.relative_to(OUTPUT).parts and path.suffix in ('.json','.csv','.txt','.log'):archive.write(path,path.relative_to(OUTPUT))
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
